# Final Submission

Лучшая модель: **LightGBM Tuned (Optuna)** — Val RMSLE = 0.3730.

**Инференс:** рекурсивный прогноз по дням (день 8+ теста требует `lag_7` из уже предсказанных продаж). Фичи считаются через `src/preprocessing/features.py` (`HistoricalData` + `compute_features`) — тот же календарь лагов/rolling, что и при обучении; `test_fe.parquet` с NaN в лагах **не используется**.

**Ячейка 2:** рекурсивный **Val RMSLE** на 2017-08-01—15 при истории только до 2017-07-31 — честная оценка режима Kaggle (сравни с ~0.3730 из parquet).

In [1]:
import os
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# корень проекта (notebooks/ → ..)
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.preprocessing.features import HistoricalData, compute_features
from src.preprocessing.transforms import (
    EncoderRegistry,
    HolidayRegistry,
    OilRegistry,
    StoreRegistry,
)

artifacts_dir = ROOT / 'artifacts'
data_dir = ROOT / 'data'
processed_dir = data_dir / 'processed'
submissions_dir = ROOT / 'submissions'
submissions_dir.mkdir(parents=True, exist_ok=True)

DROP_COLS = ('date', 'sales')
schema = pq.read_schema(processed_dir / 'train_fe.parquet')
EXPECTED_FEATURES = [c for c in schema.names if c not in DROP_COLS]

# Как в 04_advanced_models.ipynb — для совпадения с lgb.Dataset(..., categorical_feature=...)
CAT_FEATURES = [
    'store_nbr', 'family', 'city', 'state', 'type', 'cluster',
    'day_of_week', 'month', 'quarter',
]


def _feature_names_from_model(m):
    """Booster / sklearn LGBMRegressor: имена и порядок колонок как при обучении."""
    if hasattr(m, 'feature_name'):
        fn = m.feature_name
        return list(fn()) if callable(fn) else list(fn)
    b = getattr(m, 'booster_', None)
    if b is not None:
        return list(b.feature_name())
    raise TypeError('Модель без feature_name / booster_')


model = joblib.load(artifacts_dir / 'lgbm_tuned.pkl')
MODEL_FEATURES = _feature_names_from_model(model)

if set(MODEL_FEATURES) != set(EXPECTED_FEATURES):
    raise ValueError(
        f'Несовпадение фичей модели и train_fe: '
        f'model-only={set(MODEL_FEATURES) - set(EXPECTED_FEATURES)}, '
        f'parquet-only={set(EXPECTED_FEATURES) - set(MODEL_FEATURES)}'
    )

# Порядок колонок для predict — строго как в Booster (не только множество)
best_iter = getattr(model, 'best_iteration', None)
if best_iter is None:
    best_iter = getattr(model, 'best_iteration_', None)
if best_iter is None:
    best_iter = -1  # все деревья (fallback)

stores = StoreRegistry(data_dir / 'stores.csv')
oil = OilRegistry(data_dir / 'oil.csv')
holidays = HolidayRegistry(data_dir / 'holidays_events.csv')
encoders = EncoderRegistry(artifacts_dir / 'label_encoders.pkl')

test_raw = pd.read_csv(data_dir / 'test.csv')
test_raw['date'] = pd.to_datetime(test_raw['date'])

if test_raw['id'].duplicated().any():
    raise ValueError('test.csv: дублирующиеся id')
if test_raw.duplicated(subset=['date', 'store_nbr', 'family']).any():
    raise ValueError('test.csv: дубли (date, store_nbr, family)')

print(f'Features: {len(MODEL_FEATURES)} | best_iteration: {best_iter}')
print(f'Test raw: {test_raw.shape} | dates: {test_raw["date"].min().date()} — {test_raw["date"].max().date()}')
print(f'IDs: {test_raw["id"].min()} — {test_raw["id"].max()}')

Features: 30 | best_iteration: 574
Test raw: (28512, 5) | dates: 2017-08-16 — 2017-08-31
IDs: 3000888 — 3029399


In [2]:
# Рекурсивная валидация на val (2017-08-01 — 2017-08-15).
# История только до train (date <= 2017-07-31), затем по дням — как на Kaggle-тесте.
# Сравнение с офлайн ~0.3730 (parquet с реальными лагами по полному train) — 04_advanced_models.

TRAIN_END = pd.Timestamp('2017-07-31')


def rmsle(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_pred = np.clip(y_pred, 0, None)
    return float(np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2)))


train_full = pd.read_csv(data_dir / 'train.csv', parse_dates=['date'])
val_slice = train_full[
    (train_full['date'] >= '2017-08-01') & (train_full['date'] <= '2017-08-15')
].copy()

history_val = HistoricalData(data_dir / 'train.csv', max_date=TRAIN_END)

pred_val: dict[tuple, float] = {}
for d in sorted(val_slice['date'].unique()):
    day = val_slice[val_slice['date'] == d]
    rows = []
    for _, row in day.iterrows():
        feat = compute_features(
            str(row['date'].date()),
            int(row['store_nbr']),
            str(row['family']),
            int(row['onpromotion']),
            stores,
            oil,
            holidays,
            encoders,
            history_val,
        )
        rows.append(feat)

    Xv = pd.DataFrame(rows)
    Xv = Xv.reindex(columns=MODEL_FEATURES)
    for c in CAT_FEATURES:
        if c in Xv.columns:
            Xv[c] = Xv[c].astype(np.int32)
    if Xv.isna().any().any():
        raise RuntimeError(f'NaN в val (день {d.date()}): {Xv.columns[Xv.isna().any()].tolist()}')

    pred_log_v = model.predict(Xv, num_iteration=best_iter)
    if not np.isfinite(pred_log_v).all():
        raise RuntimeError(f'Неконечные pred_log на val, день {d.date()}')
    pred_sales_v = np.expm1(pred_log_v).clip(0)

    for (_, row), ps in zip(day.iterrows(), pred_sales_v):
        key = (pd.Timestamp(row['date']).normalize(), int(row['store_nbr']), str(row['family']))
        pred_val[key] = float(ps)
        history_val.record_observation(
            int(row['store_nbr']),
            str(row['family']),
            pd.Timestamp(row['date']),
            float(ps),
            int(row['onpromotion']),
        )

val_slice['pred_recursive'] = val_slice.apply(
    lambda r: pred_val[(pd.Timestamp(r['date']).normalize(), int(r['store_nbr']), str(r['family']))],
    axis=1,
)

rmsle_recursive_val = rmsle(val_slice['sales'].values, val_slice['pred_recursive'].values)
print(f'Val RMSLE (рекурсивный инференс, как на Kaggle): {rmsle_recursive_val:.4f}')
print('Офлайн val с parquet-фичами (реальные лаги): ~0.3730 — см. 04_advanced_models.ipynb')

  Loading train.csv for historical data...
  Truncated to date <= 2017-07-31 (2,981,286 rows)
  Loaded 1782 time series
Val RMSLE (рекурсивный инференс, как на Kaggle): 0.3970
Офлайн val с parquet-фичами (реальные лаги): ~0.3730 — см. 04_advanced_models.ipynb


In [3]:
# Полная история до 2017-08-15 (как при обучении фичей) — для тестового сабмита
history = HistoricalData(data_dir / 'train.csv')

pred_by_id: dict[int, float] = {}

for d in sorted(test_raw['date'].unique()):
    day = test_raw[test_raw['date'] == d]
    rows = []
    for _, row in day.iterrows():
        feat = compute_features(
            str(row['date'].date()),
            int(row['store_nbr']),
            str(row['family']),
            int(row['onpromotion']),
            stores,
            oil,
            holidays,
            encoders,
            history,
        )
        feat['id'] = int(row['id'])
        rows.append(feat)

    X = pd.DataFrame(rows).drop(columns=['id'])
    X = X.reindex(columns=MODEL_FEATURES)

    for c in CAT_FEATURES:
        if c in X.columns:
            X[c] = X[c].astype(np.int32)

    if X.isna().any().any():
        bad = X.columns[X.isna().any()].tolist()
        raise RuntimeError(f'NaN во фичах (день {d.date()}): {bad}')

    if len(X) != len(day):
        raise RuntimeError('Размер батча не совпал с test за день')

    pred_log = model.predict(X, num_iteration=best_iter)
    if not np.isfinite(pred_log).all():
        raise RuntimeError(f'Неконечные предсказания log (день {d.date()})')
    pred_sales = np.expm1(pred_log).clip(0)

    for (_, row), ps in zip(day.iterrows(), pred_sales):
        pid = int(row['id'])
        pred_by_id[pid] = float(ps)
        history.record_observation(
            int(row['store_nbr']),
            str(row['family']),
            pd.Timestamp(row['date']),
            float(ps),
            int(row['onpromotion']),
        )

submission_final = (
    test_raw[['id']]
    .assign(sales=lambda df: df['id'].map(pred_by_id))
    .sort_values('id')
)

assert submission_final['sales'].isna().sum() == 0
assert len(submission_final) == 28512
assert len(pred_by_id) == 28512
assert set(pred_by_id.keys()) == set(test_raw['id'].astype(int))

out_path = submissions_dir / 'submission.csv'
submission_final.to_csv(out_path, index=False)

print(f'Submission saved: {out_path}')
print(f'Shape: {submission_final.shape}')
print(
    f'Sales — mean: {submission_final["sales"].mean():.2f}, '
    f'min: {submission_final["sales"].min():.2f}, max: {submission_final["sales"].max():.2f}'
)
print(f'Zeros: {(submission_final["sales"] == 0).sum()} ({(submission_final["sales"] == 0).mean():.1%})')
print(submission_final.head(10))

  Loading train.csv for historical data...
  Loaded 1782 time series
Submission saved: D:\sst\submissions\submission.csv
Shape: (28512, 2)
Sales — mean: 430.53, min: 0.00, max: 15178.86
Zeros: 86 (0.3%)
        id        sales
0  3000888     4.281424
1  3000889     0.000000
2  3000890     4.987582
3  3000891  2368.245828
4  3000892     0.000000
5  3000893   409.750831
6  3000894    10.528026
7  3000895   780.045739
8  3000896   870.232301
9  3000897   153.476581
